# 16.5 节点嵌入 / Node Embeddings (DeepWalk & node2vec)

**中文**：前面我们用图算法**手算**节点的各种"分数"(度、介数、PageRank)。但要把图喂给机器学习模型(分类、聚类、推荐)，我们需要的是**每个节点一个稠密向量**——就像 Word2Vec 给每个词一个向量。**节点嵌入(node embedding)** 就是把"图结构"翻译成"向量空间"：**结构上相近的节点，嵌入向量也相近**。本节的 DeepWalk 和 node2vec 是这条路线的奠基工作，也直接复用了我们 Part 11/12 学过的 **Word2Vec skip-gram**。
**English**: We've **hand-computed** node scores (degree, betweenness, PageRank). But to feed a graph into ML models (classification, clustering, recommendation) we want **a dense vector per node** — like Word2Vec gives each word a vector. **Node embedding** translates "graph structure" into "vector space": **structurally close nodes get close embeddings**. DeepWalk and node2vec are the foundational works here, and they directly reuse the **Word2Vec skip-gram** from Part 11/12.

---

**中文**：核心洞察(DeepWalk, 2014)极其优雅——**图上的随机游走 ≈ 句子，节点 ≈ 单词**：
**English**: The core insight (DeepWalk, 2014) is elegant — **a random walk on a graph ≈ a sentence, a node ≈ a word**:

**中文**：
1. 从每个节点出发做若干次**随机游走**(每步随机走到一个邻居)，得到许多节点序列。
2. 把每条游走序列当成一个"句子"，喂给 **Word2Vec skip-gram**(用中心节点预测窗口内的上下文节点)。
3. 训练完，每个节点就有了一个向量——**经常在同一段游走里出现的节点(=结构上相近)，向量也相近**。

**English**:
1. From each node, run several **random walks** (each step hops to a random neighbor), producing many node sequences.
2. Treat each walk as a "sentence" and feed it to **Word2Vec skip-gram** (predict context nodes in a window from the center node).
3. After training, each node has a vector — **nodes that frequently co-occur in walks (= structurally close) get close vectors**.

**中文**：**node2vec(2016)** 在此基础上引入**有偏随机游走**，用两个参数 $p,q$ 控制游走"性格"：
**English**: **node2vec (2016)** adds a **biased random walk** with two parameters $p,q$ controlling the walk's "personality":
- **返回参数 $p$ (return)**：大 → 不爱走回头路(鼓励向外探索)。
  **Return param $p$**: large → avoids backtracking (encourages exploring outward).
- **进出参数 $q$ (in-out)**：$q>1$ → 倾向 **BFS**(在出发点附近打转，捕捉"结构角色/局部")；$q<1$ → 倾向 **DFS**(一路往外走，捕捉"同社区/同质性")。
  **In-out param $q$**: $q>1$ → **BFS**-like (stays near the start, captures "structural role / locality"); $q<1$ → **DFS**-like (wanders far, captures "community / homophily").

> 💡 **面试速查 / Interview cheat-sheet（★★★ GNN 前置必考）**
> **中文**：**DeepWalk = 随机游走当句子 + Word2Vec skip-gram**；**node2vec = 有偏游走(p控回头、q控BFS/DFS)**，$q{=}p{=}1$ 时退化为 DeepWalk。它们是**无监督、只用结构**的"浅层"嵌入。**三大局限(引出 GNN)**：① **不用节点特征**(只看连接); ② **直推式(transductive)**——来一个新节点必须**重训**，无法泛化(对比 GraphSAGE 的归纳式); ③ 参数=每节点一个向量, 不共享, 不scale 到超大图。评估常用**节点分类**(在嵌入上训分类器)或**链接预测**。
> **English**: **DeepWalk = random walks as sentences + Word2Vec skip-gram**; **node2vec = biased walks ($p$ controls backtracking, $q$ controls BFS/DFS)**, reducing to DeepWalk at $q{=}p{=}1$. They are **unsupervised, structure-only**, "shallow" embeddings. **Three limits (motivating GNNs)**: ① **ignore node features** (links only); ② **transductive** — a new node requires **retraining**, no generalization (vs GraphSAGE's inductive); ③ params = one vector per node, not shared, doesn't scale to huge graphs. Evaluated by **node classification** (train a classifier on embeddings) or **link prediction**.


In [ ]:

# ============================================================
# 数据集 Cora：引文网络 / Cora citation network
# 中文：2708 篇机器学习论文(节点)，5429 条引用(边)，每篇属于 7 个主题之一(标签)，
#       并带 1433 维词袋特征。引文网络的经典基准——后续 GCN/GraphSAGE/GAT 都用它。
# English: 2708 ML papers (nodes), 5429 citations (edges), each in one of 7 topics (label),
#          with 1433-dim bag-of-words features. The classic benchmark — GCN/GraphSAGE/GAT all use it.
# ============================================================
import os, time, numpy as np, torch, torch.nn as nn, torch.nn.functional as Fnn, matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
torch.manual_seed(0); np.random.seed(0)
R=os.path.expanduser("~/.cache/dsfs_recsys/cora")
content=[l.split("\t") for l in open(os.path.join(R,"cora.content")).read().strip().split("\n")]
ids=[c[0] for c in content]; id2x={v:i for i,v in enumerate(ids)}; n=len(ids)
classes=sorted(set(c[-1] for c in content)); lab2y={c:i for i,c in enumerate(classes)}
y=np.array([lab2y[c[-1]] for c in content])                       # 节点类别标签 / node labels
feats=np.array([[int(x) for x in c[1:-1]] for c in content],dtype=float)  # 词袋特征 / BoW features
# 构造无向邻接表(引用视为无向边) / undirected adjacency (citations as undirected edges)
adj=[[] for _ in range(n)]; adjset=[set() for _ in range(n)]
for line in open(os.path.join(R,"cora.cites")).read().strip().split("\n"):
    a,b=line.split("\t")
    if a in id2x and b in id2x:
        u,v=id2x[a],id2x[b]
        if v not in adjset[u]: adj[u].append(v); adjset[u].add(v); adj[v].append(u); adjset[v].add(u)
print(f"节点 {n}, 类别 {len(classes)}, 特征维 {feats.shape[1]}, 平均度 {np.mean([len(a) for a in adj]):.2f}")
print("类别 / classes:", classes)


**中文**：第一步——**生成随机游走 + 有偏游走**。普通(DeepWalk)游走每步等概率走到任一邻居；node2vec 游走则根据上一步来的方向，用 $p,q$ 给"回头/留在原地/向外"不同权重。
**English**: Step one — **generate random walks + biased walks**. A plain (DeepWalk) walk hops to any neighbor uniformly; a node2vec walk weights "backtrack / stay local / go outward" using $p,q$ based on where it came from.


In [ ]:

# ============================================================
# 随机游走(DeepWalk) 与 有偏游走(node2vec) / random & biased walks
# ============================================================
def gen_walks(p=1.0, q=1.0, num=10, length=40):
    """p=q=1 即 DeepWalk 的无偏游走; 否则为 node2vec 有偏游走 / unbiased if p=q=1, else node2vec."""
    W=[]
    for _ in range(num):                                          # 每个节点出发 num 次 / num walks per node
        for s in np.random.permutation(n):
            w=[int(s)]
            if not adj[s]: W.append(w); continue
            w.append(int(adj[s][np.random.randint(len(adj[s]))])) # 第一步随机 / first step uniform
            for _ in range(length-2):
                prev, cur = w[-2], w[-1]; nb=adj[cur]
                if not nb: break
                # node2vec 转移权重: 回到 prev=1/p; prev 的邻居(距离1)=1; 其余(距离2)=1/q
                wts=np.array([(1/p) if x==prev else (1.0 if x in adjset[prev] else 1/q) for x in nb])
                wts/=wts.sum()
                w.append(int(nb[np.random.choice(len(nb), p=wts)]))# 按偏置概率走一步 / biased step
            W.append(w)
    return W

t=time.time(); dw_walks=gen_walks(1,1); print(f"DeepWalk 游走 {len(dw_walks)} 条, 用时 {time.time()-t:.1f}s")
print("一条游走示例(前12节点)/ a walk:", dw_walks[0][:12])


**中文**：第二步——把游走当语料，**从零训练 skip-gram(负采样)**。中心节点的向量(`emb`)去预测窗口内上下文节点(`ctx`)，对采样的负样本则推开。训练完用 `emb` 当节点嵌入。这与 Part 12 的 Word2Vec 完全同源。
**English**: Step two — treat walks as a corpus and **train skip-gram (negative sampling) from scratch**. A center node's vector (`emb`) predicts its window context (`ctx`), pushing away sampled negatives. We use `emb` as the node embedding afterward. Same as Word2Vec in Part 12.


In [ ]:

# ============================================================
# 从零 skip-gram + 负采样 / skip-gram with negative sampling from scratch
# ============================================================
deg=np.array([len(a) for a in adj])+1.0; noise=deg**0.75; noise/=noise.sum()   # 负采样分布 ∝ 度^0.75
def train_skipgram(W, win=4, d=64, neg=5, epochs=2, sub=3_000_000, seed=0):
    torch.manual_seed(seed); np.random.seed(seed)
    pairs=[]                                                       # (中心,上下文) 对 / (center,context) pairs
    for w in W:
        for i,c in enumerate(w):
            for j in range(max(0,i-win), min(len(w),i+win+1)):
                if i!=j: pairs.append((c,w[j]))
    pairs=np.array(pairs)
    emb=nn.Embedding(n,d); ctx=nn.Embedding(n,d)
    nn.init.uniform_(emb.weight,-0.5/d,0.5/d); nn.init.zeros_(ctx.weight)
    opt=torch.optim.Adam(list(emb.parameters())+list(ctx.parameters()), lr=0.01)
    for ep in range(epochs):
        idx=np.random.permutation(len(pairs))[:sub]                # 每轮子采样加速 / subsample per epoch
        for b in range(0,len(idx),4096):
            bi=idx[b:b+4096]; ce=torch.tensor(pairs[bi,0]); co=torch.tensor(pairs[bi,1])
            ng=torch.tensor(np.random.choice(n,size=(len(bi),neg),p=noise))   # 负样本 / negatives
            ve=emb(ce); vp=ctx(co); vn=ctx(ng)
            loss=-(Fnn.logsigmoid((ve*vp).sum(1)) +                # 正样本拉近 / pull positive
                   Fnn.logsigmoid(-(vn*ve.unsqueeze(1)).sum(2)).sum(1)).mean()  # 负样本推开 / push negatives
            opt.zero_grad(); loss.backward(); opt.step()
    return emb.weight.detach().numpy()

t=time.time(); X_dw=train_skipgram(dw_walks); print(f"DeepWalk 嵌入训练完 / trained in {time.time()-t:.0f}s, shape {X_dw.shape}")


**中文**：第三步——**评估嵌入质量**。最常用的是**节点分类**：把节点嵌入当特征，训练一个逻辑回归预测论文主题。我们对比 DeepWalk、node2vec(两种 $p,q$)、以及**原始词袋特征**基线——看"纯结构嵌入"能否打过"纯文本特征"。
**English**: Step three — **evaluate embedding quality** via **node classification**: use node embeddings as features to train a logistic regression predicting the paper's topic. We compare DeepWalk, node2vec (two $p,q$ settings), and the **raw bag-of-words** baseline — does "structure-only embedding" beat "text-only features"?


In [ ]:

# ============================================================
# 节点分类评估 + node2vec 两种偏置 / node classification + node2vec settings
# ============================================================
np.random.seed(1); perm=np.random.permutation(n); tr=perm[:1400]; te=perm[1400:]   # 训练/测试划分
def clf_acc(X): return LogisticRegression(max_iter=1000).fit(X[tr],y[tr]).score(X[te],y[te])

X_n2v_dfs=train_skipgram(gen_walks(p=1,q=0.5))   # DFS 倾向(同质性/社区) / DFS-like (homophily)
X_n2v_bfs=train_skipgram(gen_walks(p=1,q=2.0))   # BFS 倾向(结构角色/局部) / BFS-like (structural)
res={"DeepWalk (p=q=1)":clf_acc(X_dw),
     "node2vec (q=0.5, DFS)":clf_acc(X_n2v_dfs),
     "node2vec (q=2, BFS)":clf_acc(X_n2v_bfs),
     "原始词袋特征 raw BoW":clf_acc(feats)}
print(f"{'方法/method':<26}{'节点分类准确率 / accuracy':>22}")
for k,v in res.items(): print(f"{k:<26}{v:>22.4f}")


In [ ]:

# ============================================================
# 可视化：t-SNE 把 64 维嵌入压到 2D, 按真实主题上色 / t-SNE of embeddings colored by topic
# ============================================================
from sklearn.manifold import TSNE
fig,ax=plt.subplots(1,2,figsize=(15,6))
pal=plt.cm.tab10(np.linspace(0,1,len(classes)))
for axi,(name,X) in zip(ax,[("DeepWalk 嵌入 / embedding",X_dw),("原始词袋特征 / raw BoW",feats)]):
    Z=TSNE(n_components=2,init="pca",random_state=0,perplexity=30).fit_transform(X)
    for ci in range(len(classes)):
        mk=y==ci; axi.scatter(Z[mk,0],Z[mk,1],s=8,color=pal[ci],label=classes[ci][:12],alpha=0.7)
    axi.set_title(name); axi.set_xticks([]); axi.set_yticks([])
ax[0].legend(fontsize=6,markerscale=2,loc="best")
plt.tight_layout(); plt.savefig("/tmp/g05_viz.png",dpi=80); plt.show()
print("DeepWalk 嵌入里同主题论文自然聚成簇——而它只看了引用结构, 没看论文内容!")
print("DeepWalk embeddings cluster same-topic papers — using only citation structure, not text!")


**中文**：诚实解读：
**English**: Honest takeaways:

**中文**：
1. **纯结构嵌入打败了纯文本特征**：DeepWalk(只用引用关系、完全没看论文内容)的节点分类准确率(~0.78)**高于** 1433 维词袋特征(~0.74)。这说明**"谁引用谁"比"论文写了什么词"更能预测主题**——同主题论文倾向于互相引用(同质性)。这是图结构信息价值的有力证据。
2. **node2vec 的偏置只带来微小差异**：在 Cora 上三种设置都在 ~0.78–0.80，彼此**差异仅 ~1-2%、且在随机噪声范围内**。其中 **DFS 倾向(q=0.5)** 往往略占优——这与理论一致：$q<1$ 偏向 DFS、强调**同质性/社区**，而引文网络正是强同质的(同主题论文互引)。诚实地说——**在这种强同质网络上，复杂的游走偏置收益有限**；node2vec 的优势在需要区分"结构角色"的任务(如识别桥梁节点)上更明显。别迷信"更复杂就更好"。
3. **t-SNE 可视化**：DeepWalk 嵌入里 7 个主题自然分成可见的簇——再次强调，**它从未读过论文一个字，只看了引用图**。

**English**:
1. **Structure-only embeddings beat text-only features**: DeepWalk (citation links only, never reading content) classifies topics at ~0.78, **above** the 1433-dim bag-of-words (~0.74). So **"who cites whom" predicts topic better than "what words the paper uses"** — same-topic papers tend to cite each other (homophily). Strong evidence for the value of graph structure.
2. **node2vec's bias gives only marginal differences**: on Cora all three settings sit at ~0.78–0.80, differing by only ~1–2% and **within random noise**. **DFS-like (q=0.5)** tends to edge ahead — consistent with theory: $q<1$ favors DFS and emphasizes **homophily/community**, and citation networks are strongly homophilous (same-topic papers cite each other). Honestly — **on such a homophilous network, fancy walk biasing brings limited gains**; node2vec's edge shows more on tasks needing "structural roles" (e.g. identifying bridges). Don't assume "more complex = better."
3. **t-SNE visualization**: the 7 topics form visible clusters in DeepWalk space — again, **it never read a single word, only the citation graph**.

> 💼 **实战视角 / Practical angle**
> **中文**：节点嵌入用于:推荐(用户/物品图嵌入做召回, 阿里 EGES)、反欺诈(账号关系图嵌入找异常)、链接预测(预测两节点会不会连边)、可视化。**最大软肋是"直推式"**:DeepWalk/node2vec 给每个**已知**节点学一个向量, **新节点来了必须重训**——在每天新增百万节点的工业图上不可接受。这正是下一节起 **GNN(GCN/GraphSAGE/GAT)** 要解决的:用**可泛化的聚合函数**(而非每节点独立向量) + **融合节点特征**, 实现**归纳式**嵌入。面试金句:*"DeepWalk 把图变语料、随机游走当句子、复用 Word2Vec; 但它是直推式、不用特征的浅层方法, GNN 是它的归纳式、用特征的深层继承者。"*
> **English**: Node embeddings power: recommendation (user/item-graph embeddings for retrieval, Alibaba's EGES), anti-fraud (account-relation-graph embeddings to spot anomalies), link prediction, visualization. **The biggest weakness is "transductive"**: DeepWalk/node2vec learn one vector per **known** node, so a **new node forces retraining** — unacceptable on industrial graphs gaining millions of nodes daily. This is exactly what **GNNs (GCN/GraphSAGE/GAT)** fix from the next section: a **generalizable aggregation function** (not per-node vectors) + **fusing node features** for **inductive** embeddings. Interview line: *"DeepWalk turns a graph into a corpus, walks into sentences, reusing Word2Vec; but it's transductive and feature-free and shallow — GNNs are its inductive, feature-using, deep successors."*

---
### 小结 / Summary
- **中文**：节点嵌入=把图结构编码成稠密向量；DeepWalk=随机游走+skip-gram；node2vec=有偏游走(p/q 调 BFS/DFS)。
- **English**: Node embeddings encode structure into dense vectors; DeepWalk = random walks + skip-gram; node2vec = biased walks (p/q tune BFS/DFS).
- **中文**：Cora 上纯结构嵌入(~0.78)胜过纯文本特征(~0.74)——引用关系比用词更能预测主题。
- **English**: On Cora, structure-only embeddings (~0.78) beat text-only features (~0.74) — citations predict topic better than words.
- **中文**：局限:直推式(新节点要重训)+不用特征+浅层——直接引出 GNN。
- **English**: Limits: transductive (retrain for new nodes) + feature-free + shallow — directly motivating GNNs.
